In [1]:
! uv pip install seaborn morethemes pyfonts pypalettes drawarrow dayplot contextily geopandas tqdm imageio

Using Python 3.13.9 environment at: C:\Users\Asus\Documents\14_PrixChangementClimatique\.venv
Resolved 43 packages in 1.15s
Prepared 2 packages in 298ms
Installed 13 packages in 272ms
 + affine==2.4.0
 + cligj==0.7.2
 + contextily==1.7.0
 + dayplot==0.5.1
 + drawarrow==0.1.0
 + geographiclib==2.1
 + geopy==2.4.1
 + imageio==2.37.3
 + mercantile==1.2.1
 + morethemes==0.5.1
 + pyfonts==1.3.0
 + pypalettes==0.2.1
 + rasterio==1.5.0


# IMPORT


In [3]:
import duckdb

# The usual suspects
import numpy as np
import pandas as pd

# Chart and Figure
import seaborn as sns
import matplotlib
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.ticker as ticker
from matplotlib.colors import LinearSegmentedColormap

# Fonts, Palettes, Themes, Arrow (https://github.com/y-sunflower)
# Strong inspiration: https://www.yan-holtz.com/
import morethemes as mt
from pyfonts import load_google_font, load_font
from pypalettes import load_palette, load_cmap
from drawarrow import fig_arrow
import dayplot as dp  # Calendar plot for timeseries

# GeoSpatial
import contextily as ctx
from shapely.affinity import translate, scale
import geopandas as gpd

# GIF
import imageio
from tqdm import tqdm

# Removing warnings
import warnings

warnings.filterwarnings("ignore")

# LOAD DATA


In [5]:
# Setup
font = load_google_font(
    "Montserrat", italic=False
)  # check here: https://fonts.google.com/
# mt.set_theme("nature")

source_text = (
    "Source: CCR (Caisse Centrale de Réassurance)"  # Main Source for the analysis
)

In [18]:
communes_geojson_url = "https://raw.githubusercontent.com/gregoiredavid/france-geojson/master/communes-avec-outre-mer.geojson"
communes_geo = gpd.read_file(communes_geojson_url)

In [ ]:
con = duckdb.connect(database="dev.duckdb", read_only=True)

In [14]:
query_ccr_details = """
					SELECT
						*
					FROM dev.main.ccr_details
					"""

ccr_details = con.sql(query_ccr_details)
ccr_details_df = ccr_details.df()
ccr_details_df = ccr_details_df[
    ~ccr_details_df.duplicated(subset=["code_geo", "code_arrete"])
]
ccr_details_df.head(2)

,code_geo,nom_commune,date_debut_evenement,date_fin_evenement,date_arrete,date_parution_jo,nom_peril,franchise,libelle_avis,code_arrete
0,07005,ALBA LA ROMAINE,2025-11-16,2025-11-16,2026-01-19,2026-01-24,Inondations et/ou Coulées de Boue,-,Non reconnue,904
1,07022,BAIX,2025-11-16,2025-11-17,2026-01-19,2026-01-24,Inondations et/ou Coulées de Boue,Simple,Reconnue,904


In [ ]:
details_par_catnat = ccr_details_df.copy()

details_par_catnat["nom_peril"] = (
    details_par_catnat["nom_peril"]
    .mask(details_par_catnat["nom_peril"].str.contains("Inondations"), "Inondations")
    .where(
        details_par_catnat["nom_peril"].str.contains("Inondations|Sécheresse"), "Autre"
    )
)

details_par_catnat = details_par_catnat[
    (details_par_catnat["date_arrete"].dt.year > 1983)
    & (details_par_catnat["date_arrete"].dt.year < 2026)
]
details_par_catnat["periode_2010"] = np.where(
    details_par_catnat["date_arrete"].dt.year < 2010, "Avant 2010", "Après 2010"
)

avis_par_catnat = (
    details_par_catnat.groupby(
        [
            pd.Grouper(key="date_arrete", freq="YE"),
            "nom_peril",
            "libelle_avis",
            "periode_2010",
        ]
    )
    .size()
    .reset_index(name="count")
)